# Colab DINOv2 Feature Extraction for Experiment 1

This notebook runs frozen DINOv2 feature extraction on Google Colab using a compressed `data/exp1_bounded` archive stored in Google Drive.

It is designed for the storage setup where the compressed archive fits in Drive, but the uncompressed dataset does not. The archive is extracted onto Colab local disk, feature caches are computed on the GPU, and compact `.npz` outputs are copied back to Drive.

Before running:

1. In Colab, choose **Runtime -> Change runtime type -> GPU**.
2. Upload this notebook to Colab.
3. Set `DRIVE_ARCHIVE` below to your compressed `exp1_bounded` archive path in Drive.
4. Provide the repo through either `REPO_URL` or `REPO_ZIP_IN_DRIVE`.

In [ ]:
from pathlib import Path

# Required: path to the compressed data/exp1_bounded archive in Google Drive.
# Examples:
#   /content/drive/MyDrive/exp1_bounded.tar.gz
#   /content/drive/MyDrive/exp1_bounded.zip
DRIVE_ARCHIVE = "/content/drive/MyDrive/exp1_bounded.tar.gz"

# Required unless the repo already exists at REPO_DIR.
# Use REPO_URL for a GitHub remote, or REPO_ZIP_IN_DRIVE for a zipped repo in Drive.
REPO_URL = ""  # e.g. "https://github.com/USER/cv-project.git"
REPO_ZIP_IN_DRIVE = ""  # e.g. "/content/drive/MyDrive/cv-project.zip"
FORCE_REPO_REFRESH = False  # Set true if Colab is reusing an old repo copy.

# Persistent feature outputs go here. Chunk outputs are copied as they finish.
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/exp1_bounded_dinov2_features"

# Local Colab workspace. This is fast but ephemeral.
WORK_DIR = Path("/content/exp1_colab")
REPO_DIR = WORK_DIR / "cv-project"
DATA_EXTRACT_DIR = WORK_DIR / "archive_extract"
DATA_ROOT = WORK_DIR / "data" / "exp1_bounded"
FEATURE_WORK_DIR = WORK_DIR / "features"
COLAB_MANIFEST = WORK_DIR / "manifests" / "render_valid_colab.parquet"

# DINOv2 feature extraction settings.
MODEL_NAMES = ["dinov2_vit_b"]
LAYERS = ["final", "layer4", "layer8", "layer12"]
BATCH_SIZE = 8       # Try 16 on L4/A100. Use 4 if a T4 runs out of memory.
NUM_WORKERS = 4      # Image preprocessing workers.

# Optional filters. Empty lists mean all rows.
SPLITS = []  # e.g. ["train"]
TEXTURE_CONDITIONS = []  # e.g. ["flat", "random_noise"]

# Chunking makes the run resumable and protects progress if Colab disconnects.
ROWS_PER_JOB = 2048  # Set to None to run all selected rows as one job.
START_JOB = 0
MAX_JOBS = None      # e.g. 2 for a short test run.
RESUME = True

print("Configured workspace:", WORK_DIR)

## Mount Drive and Check GPU

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess
import sys
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No CUDA GPU is available. In Colab, switch Runtime type to GPU.")

subprocess.run(["nvidia-smi"], check=False)

## Install Python Dependencies

This installs only the feature-extraction dependencies, not Blender or rendering packages.

In [ ]:
packages = [
    "transformers>=4.38.0",
    "huggingface_hub>=0.20.0",
    "open_clip_torch>=2.24.0",
    "hydra-core>=1.3.2",
    "omegaconf>=2.3.0",
    "pandas>=2.0.0",
    "pyarrow>=14.0.0",
    "Pillow>=10.0.0",
    "tqdm>=4.66.0",
    "scikit-learn>=1.3.0",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependencies installed.")

## Prepare Repo

If `REPO_URL` is set, the notebook clones it. If `REPO_ZIP_IN_DRIVE` is set, the notebook extracts it and finds the repo root by looking for `scripts/extract_exp1_features.py`.

In [ ]:
import json
import shutil
import tarfile
import zipfile

def run(cmd, *, cwd=None, env=None):
    printable = " ".join(str(x) for x in cmd)
    print("+", printable)
    result = subprocess.run(
        [str(x) for x in cmd],
        cwd=cwd,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr, file=sys.stderr)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {printable}")
    return result

def safe_extract_archive(archive_path: Path, dest: Path) -> None:
    archive_path = Path(archive_path)
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    resolved_dest = dest.resolve()
    print(f"Extracting {archive_path} -> {dest}")
    if archive_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(archive_path) as zf:
            for member in zf.infolist():
                target = (dest / member.filename).resolve()
                if not str(target).startswith(str(resolved_dest)):
                    raise RuntimeError(f"Unsafe archive member: {member.filename}")
            zf.extractall(dest)
        return
    if tarfile.is_tarfile(archive_path):
        with tarfile.open(archive_path) as tf:
            for member in tf.getmembers():
                target = (dest / member.name).resolve()
                if not str(target).startswith(str(resolved_dest)):
                    raise RuntimeError(f"Unsafe archive member: {member.name}")
            tf.extractall(dest)
        return
    raise ValueError(f"Unsupported archive format: {archive_path}")

def find_repo_root(root: Path) -> Path:
    hits = sorted(root.rglob("scripts/extract_exp1_features.py"))
    if not hits:
        raise FileNotFoundError("Could not find scripts/extract_exp1_features.py in extracted repo")
    return hits[0].parents[1]

def ensure_exp1_data_package(repo_dir: Path) -> None:
    """Create the small exp1.data package if an older clone omitted it."""
    package_dir = repo_dir / "exp1" / "data"
    splits_path = package_dir / "splits.py"
    if splits_path.is_file():
        return
    print("Creating missing exp1.data shim:", package_dir)
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / "__init__.py").write_text(
        '"""Data utilities for Experiment 1."""\n',
        encoding="utf-8",
    )
    splits_path.write_text(
        '''"""Split filtering utilities for Experiment 1."""\n\nfrom __future__ import annotations\n\nfrom typing import Iterable, Mapping, Optional, Sequence, Union\n\nimport pandas as pd\n\nRowsLike = Union[pd.DataFrame, Iterable[Mapping[str, object]]]\n\n\ndef _as_dataframe(rows: RowsLike) -> pd.DataFrame:\n    if isinstance(rows, pd.DataFrame):\n        return rows.copy()\n    return pd.DataFrame(list(rows))\n\n\ndef filter_manifest(\n    rows: RowsLike,\n    *,\n    split: Optional[Union[str, Sequence[str]]] = None,\n    texture_condition: Optional[Union[str, Sequence[str]]] = None,\n    require_qc_pass: bool = False,\n    require_label_valid: bool = False,\n) -> pd.DataFrame:\n    df = _as_dataframe(rows)\n    if split is not None:\n        allowed = {str(split)} if isinstance(split, str) else {str(v) for v in split}\n        df = df[df["split"].astype(str).isin(allowed)]\n    if texture_condition is not None:\n        allowed = (\n            {str(texture_condition)}\n            if isinstance(texture_condition, str)\n            else {str(v) for v in texture_condition}\n        )\n        df = df[df["texture_condition"].astype(str).isin(allowed)]\n    if require_qc_pass and "qc_pass" in df.columns:\n        df = df[df["qc_pass"].astype(bool)]\n    if require_label_valid and "label_valid" in df.columns:\n        df = df[df["label_valid"].astype(bool)]\n    return df.reset_index(drop=True)\n''',
        encoding="utf-8",
    )

WORK_DIR.mkdir(parents=True, exist_ok=True)
if FORCE_REPO_REFRESH and REPO_DIR.exists():
    print("Removing existing repo because FORCE_REPO_REFRESH=True:", REPO_DIR)
    shutil.rmtree(REPO_DIR)

if (REPO_DIR / "scripts" / "extract_exp1_features.py").is_file():
    print("Repo already exists:", REPO_DIR)
elif REPO_URL.strip():
    run(["git", "clone", "--depth", "1", REPO_URL.strip(), REPO_DIR])
elif REPO_ZIP_IN_DRIVE.strip():
    repo_extract_dir = WORK_DIR / "repo_extract"
    safe_extract_archive(Path(REPO_ZIP_IN_DRIVE), repo_extract_dir)
    found = find_repo_root(repo_extract_dir)
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    shutil.move(str(found), str(REPO_DIR))
else:
    raise RuntimeError("Set REPO_URL or REPO_ZIP_IN_DRIVE, or place the repo at REPO_DIR.")

ensure_exp1_data_package(REPO_DIR)

sys.path.insert(0, str(REPO_DIR))
os.environ["CV_PROJECT_ROOT"] = str(REPO_DIR)
os.environ.setdefault("HF_HOME", str(WORK_DIR / "hf_cache"))
os.environ["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")
print("Repo ready:", REPO_DIR)
print("HF_HOME:", os.environ["HF_HOME"])

help_result = run(
    [sys.executable, str(REPO_DIR / "scripts" / "extract_exp1_features.py"), "--help"],
    cwd=REPO_DIR,
    env=os.environ.copy(),
)
help_text = (help_result.stdout or "") + (help_result.stderr or "")
required_flags = ["--batch-size", "--num-workers", "--allow-unvalidated"]
missing_flags = [flag for flag in required_flags if flag not in help_text]
if missing_flags:
    raise RuntimeError(
        "The repo copy in Colab is missing expected extraction flags: "
        f"{missing_flags}. Set FORCE_REPO_REFRESH=True and rerun this cell, "
        "or upload/clone the latest repo."
    )

## Extract `exp1_bounded` to Local Disk

Do not extract the dataset into mounted Drive. Mounted Drive is much slower for thousands of PNG reads and may not have enough uncompressed space.

In [ ]:
def find_exp1_bounded_root(root: Path) -> Path:
    candidates = []
    if root.name == "exp1_bounded" and (root / "renders").exists():
        candidates.append(root)
    candidates.extend(
        p for p in root.rglob("exp1_bounded")
        if p.is_dir() and (p / "renders").exists() and (p / "manifests").exists()
    )
    if not candidates:
        raise FileNotFoundError("Could not find an extracted exp1_bounded directory")
    return sorted(candidates, key=lambda p: len(str(p)))[0]

archive_path = Path(DRIVE_ARCHIVE)
if not archive_path.is_file():
    raise FileNotFoundError(f"Missing DRIVE_ARCHIVE: {archive_path}")

if DATA_ROOT.exists() and (DATA_ROOT / "renders").exists():
    print("Using existing local data root:", DATA_ROOT)
else:
    if DATA_EXTRACT_DIR.exists():
        print("Using existing extraction directory:", DATA_EXTRACT_DIR)
    else:
        safe_extract_archive(archive_path, DATA_EXTRACT_DIR)
    extracted_root = find_exp1_bounded_root(DATA_EXTRACT_DIR)
    DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if DATA_ROOT.exists() or DATA_ROOT.is_symlink():
        DATA_ROOT.unlink() if DATA_ROOT.is_symlink() else shutil.rmtree(DATA_ROOT)
    os.symlink(extracted_root, DATA_ROOT, target_is_directory=True)

# Make the repo config path REPO_DIR/data/exp1_bounded resolve to the local data.
repo_data_path = REPO_DIR / "data" / "exp1_bounded"
repo_data_path.parent.mkdir(parents=True, exist_ok=True)
if not repo_data_path.exists() and not repo_data_path.is_symlink():
    os.symlink(DATA_ROOT, repo_data_path, target_is_directory=True)

print("Local data root:", DATA_ROOT.resolve())
print("Repo data path:", repo_data_path)

## Build a Colab Manifest

Your render status files contain absolute paths from the local machine. This cell rewrites any `data/exp1_bounded/...` paths to the Colab local extraction path and writes a feature-extraction manifest.

In [ ]:
import pandas as pd

def read_manifest(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return pd.DataFrame(rows)
    raise ValueError(f"Unsupported manifest type: {path}")

manifest_dir = DATA_ROOT / "manifests"
manifest_candidates = [
    manifest_dir / "render_valid.parquet",
    manifest_dir / "render_valid.jsonl",
    manifest_dir / "render_qc.parquet",
    manifest_dir / "render_qc.jsonl",
]

df = None
for candidate in manifest_candidates:
    if candidate.is_file():
        print("Loading manifest:", candidate)
        df = read_manifest(candidate)
        break

if df is None:
    status_files = sorted((manifest_dir / "render_chunks").glob("*.render_status.jsonl"))
    if not status_files:
        raise FileNotFoundError("No render_valid/render_qc manifest or chunk status JSONL files found")
    rows = []
    for status_file in status_files:
        with status_file.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    print(f"Built manifest from {len(status_files)} render status files")

if "render_status" in df.columns:
    df = df[df["render_status"].astype(str).str.lower().isin(["success", "ok", "complete", "completed"])]
if "qc_pass" in df.columns:
    df = df[df["qc_pass"].astype(bool)]
if SPLITS:
    df = df[df["split"].astype(str).isin(SPLITS)]
if TEXTURE_CONDITIONS:
    df = df[df["texture_condition"].astype(str).isin(TEXTURE_CONDITIONS)]

def remap_exp1_path(value):
    if pd.isna(value):
        return value
    text = str(value)
    marker = "data/exp1_bounded/"
    if marker in text:
        rel = text.split(marker, 1)[1]
        return str((DATA_ROOT / rel).resolve())
    return text

path_columns = [c for c in df.columns if c.endswith("_path")]
for col in path_columns:
    df[col] = df[col].map(remap_exp1_path)

if "render_id" not in df.columns or "rgb_path" not in df.columns:
    raise ValueError("Manifest must contain render_id and rgb_path columns")
df = df.drop_duplicates(subset=["render_id"]).reset_index(drop=True)

exists_mask = df["rgb_path"].map(lambda p: Path(str(p)).is_file())
if not exists_mask.all():
    missing = df.loc[~exists_mask, ["render_id", "rgb_path"]].head(10)
    raise FileNotFoundError(f"Missing RGB files after path remap:\n{missing}")

COLAB_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(COLAB_MANIFEST, index=False)
print(f"Wrote {len(df)} valid rows to {COLAB_MANIFEST}")
display(df[["render_id", "split", "texture_condition", "rgb_path"]].head())
display(df.groupby(["split", "texture_condition"]).size().rename("rows").reset_index())

## Run Chunked DINOv2 Extraction

Each chunk is copied to Drive immediately after it finishes. If Colab disconnects, rerun the notebook with `RESUME = True`; completed chunks will be skipped.

In [ ]:
import math

torch.set_float32_matmul_precision("high")
FEATURE_WORK_DIR.mkdir(parents=True, exist_ok=True)
drive_output = Path(DRIVE_OUTPUT_DIR)
drive_chunks_dir = drive_output / "chunks"
drive_chunks_dir.mkdir(parents=True, exist_ok=True)

selected = pd.read_parquet(COLAB_MANIFEST).reset_index(drop=True)
if ROWS_PER_JOB is None:
    slices = [(0, len(selected))]
else:
    slices = [(i, min(i + int(ROWS_PER_JOB), len(selected))) for i in range(0, len(selected), int(ROWS_PER_JOB))]

jobs = list(enumerate(slices))
jobs = [(idx, slc) for idx, slc in jobs if idx >= int(START_JOB)]
if MAX_JOBS is not None:
    jobs = jobs[: int(MAX_JOBS)]
print(f"Selected {len(selected)} rows across {len(slices)} total jobs; running {len(jobs)} jobs")

def expected_npz_paths(base_dir: Path):
    return [base_dir / model / f"{layer}.npz" for model in MODEL_NAMES for layer in LAYERS]

env = os.environ.copy()
env["CV_PROJECT_ROOT"] = str(REPO_DIR)
env["HF_HOME"] = os.environ["HF_HOME"]
env["PYTHONPATH"] = str(REPO_DIR) + os.pathsep + env.get("PYTHONPATH", "")

for job_idx, (start, stop) in jobs:
    job_name = f"job_{job_idx:05d}_{start:07d}_{stop:07d}"
    job_manifest = WORK_DIR / "manifests" / "feature_jobs" / f"{job_name}.parquet"
    local_out = FEATURE_WORK_DIR / job_name
    drive_out = drive_chunks_dir / job_name
    job_manifest.parent.mkdir(parents=True, exist_ok=True)
    selected.iloc[start:stop].to_parquet(job_manifest, index=False)

    if RESUME and drive_out.exists() and not all(p.is_file() for p in expected_npz_paths(local_out)):
        print(f"Restoring completed Drive chunk for {job_name}")
        shutil.copytree(drive_out, local_out, dirs_exist_ok=True)

    if RESUME and all(p.is_file() for p in expected_npz_paths(local_out)):
        print(f"Skipping completed {job_name}")
        continue

    cmd = [
        sys.executable,
        str(REPO_DIR / "scripts" / "extract_exp1_features.py"),
        "--config", str(REPO_DIR / "configs" / "exp1_bounded.yaml"),
        "--render-manifest", str(job_manifest),
        "--feature-dir", str(local_out),
        "--models", *MODEL_NAMES,
        "--layers", *LAYERS,
        "--device", "cuda",
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--allow-unvalidated",
    ]
    run(cmd, cwd=REPO_DIR, env=env)
    if not all(p.is_file() for p in expected_npz_paths(local_out)):
        missing = [str(p) for p in expected_npz_paths(local_out) if not p.is_file()]
        raise RuntimeError(f"Extraction finished but outputs are missing: {missing}")
    shutil.copytree(local_out, drive_out, dirs_exist_ok=True)
    print(f"Copied {job_name} outputs to {drive_out}")

print("Chunk extraction complete.")

## Merge Chunk Outputs

This writes canonical feature caches under `DRIVE_OUTPUT_DIR/merged/<model>/<layer>.npz`. These files preserve `render_id` arrays and can be copied back into `data/exp1_bounded/features` for probe training.

In [ ]:
import numpy as np

sys.path.insert(0, str(REPO_DIR))
from exp1.features.storage import load_feature_cache, save_feature_cache

chunk_dirs = sorted(p for p in drive_chunks_dir.glob("job_*") if p.is_dir())
if not chunk_dirs:
    chunk_dirs = sorted(p for p in FEATURE_WORK_DIR.glob("job_*") if p.is_dir())
if not chunk_dirs:
    raise RuntimeError("No chunk outputs found to merge")

merged_dir = drive_output / "merged"
merged_dir.mkdir(parents=True, exist_ok=True)

for model in MODEL_NAMES:
    for layer in LAYERS:
        all_ids = []
        all_features = []
        used_chunks = []
        for chunk_dir in chunk_dirs:
            cache_path = chunk_dir / model / f"{layer}.npz"
            if not cache_path.is_file():
                continue
            cache = load_feature_cache(cache_path)
            all_ids.extend([str(x) for x in cache["render_ids"].tolist()])
            all_features.append(cache["features"])
            used_chunks.append(chunk_dir.name)
        if not all_features:
            raise RuntimeError(f"No feature chunks found for {model}/{layer}")
        if len(set(all_ids)) != len(all_ids):
            raise RuntimeError(f"Duplicate render_id values while merging {model}/{layer}")
        features = np.concatenate(all_features, axis=0).astype(np.float32)
        out_path = merged_dir / model / f"{layer}.npz"
        save_feature_cache(
            out_path,
            render_ids=all_ids,
            features=features,
            metadata={
                "model_name": model,
                "layer_name": layer,
                "token": "cls",
                "feature_type": "global",
                "normalized": True,
                "source_chunks": used_chunks,
            },
        )
        print(f"Merged {model}/{layer}: {features.shape} -> {out_path}")

print("Merged feature caches are in:", merged_dir)

## After Colab Finishes

Copy the files from `DRIVE_OUTPUT_DIR/merged` back into the project as `data/exp1_bounded/features` or pass that merged directory as the feature directory for downstream probe training.

If Colab disconnects before the merge cell, rerun the notebook with the same `DRIVE_OUTPUT_DIR`. The extraction cell will restore and skip completed chunks, then the merge cell will combine whatever chunks are present.